In [2]:
import ast
import json
import sys
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, packages_distributions, version
from pathlib import Path

ROOT = Path("..").resolve()
NOTEBOOKS_DIR = ROOT / "notebooks_code"
REQUIREMENTS_TXT = ROOT / "requirements.txt"

# Discover every notebook inside notebooks_code automatically.
NOTEBOOKS = sorted(
    p for p in NOTEBOOKS_DIR.rglob("*.ipynb")
    if "ipynb_checkpoints" not in p.parts
)

stdlib = set(getattr(sys, "stdlib_module_names", set()))
module_to_distributions = packages_distributions()

def imported_top_modules(code: str) -> set[str]:
    modules = set()
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return modules

    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                modules.add(alias.name.split(".")[0])
        elif isinstance(node, ast.ImportFrom) and node.module:
            modules.add(node.module.split(".")[0])
    return modules

def notebook_code_sources(nb_path: Path) -> list[str]:
    nb = json.loads(nb_path.read_text(encoding="utf-8"))
    sources = []
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        src = "\n".join(cell.get("source", []))
        sources.append(src)
    return sources

def infer_distribution_name(module: str) -> str | None:
    # Preferred path: use installed metadata mapping from top-level module to distribution(s).
    distributions = module_to_distributions.get(module)
    if distributions:
        return sorted(distributions)[0]

    # Fallback: in many cases, distribution name matches module name.
    try:
        version(module)
        return module
    except PackageNotFoundError:
        return None

used_modules = set()
uses_xlsx_excel_io = False

for nb_path in NOTEBOOKS:
    for src in notebook_code_sources(nb_path):
        used_modules.update(imported_top_modules(src))
        if "read_excel(" in src and ".xlsx" in src:
            uses_xlsx_excel_io = True

packages = set()
unresolved_modules = []

for module in sorted(used_modules):
    if module in stdlib:
        continue
    dist_name = infer_distribution_name(module)
    if dist_name is None:
        unresolved_modules.append(module)
        continue
    packages.add(dist_name)

# pandas.read_excel on .xlsx requires openpyxl engine at runtime.
if uses_xlsx_excel_io:
    packages.add("openpyxl")

missing_versions = []
pinned_requirements = []
for pkg in sorted(packages):
    try:
        pinned_requirements.append(f"{pkg}=={version(pkg)}")
    except PackageNotFoundError:
        missing_versions.append(pkg)

if missing_versions:
    raise RuntimeError(
        "Cannot pin versions for: " + ", ".join(missing_versions) +
        ". Install them first, then rerun this cell."
    )

generated_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
header = [
    "# Auto-generated from notebook imports",
    "# Source notebooks:",
    *[f"# - {p.relative_to(ROOT).as_posix()}" for p in NOTEBOOKS],
    f"# Generated at: {generated_at}",
    "",
]

content = "\n".join(header + pinned_requirements) + "\n"
REQUIREMENTS_TXT.write_text(content, encoding="utf-8")

print("Notebooks scanned:")
for p in NOTEBOOKS:
    print(f"- {p}")

if unresolved_modules:
    print("\nWarning: unresolved third-party modules (skipped):")
    for m in sorted(unresolved_modules):
        print(f"- {m}")

print("\nPinned packages written to requirements.txt:")
for req in pinned_requirements:
    print(f"- {req}")

print(f"\nUpdated: {REQUIREMENTS_TXT}")

Notebooks scanned:
- /workspaces/ai4as_pitfalls_in_machine_learning_2026/notebooks_code/99_update_requirements.ipynb
- /workspaces/ai4as_pitfalls_in_machine_learning_2026/notebooks_code/class_01_exercise.ipynb
- /workspaces/ai4as_pitfalls_in_machine_learning_2026/notebooks_code/class_02_exercise.ipynb
- /workspaces/ai4as_pitfalls_in_machine_learning_2026/notebooks_code/class_03_exercise.ipynb

Pinned packages written to requirements.txt:
- Boruta==0.4.3
- imbalanced-learn==0.14.2
- matplotlib==3.10.9
- numpy==2.4.4
- openpyxl==3.1.5
- pandas==3.0.3
- scikit-learn==1.8.0
- seaborn==0.13.2
- statsmodels==0.14.6

Updated: /workspaces/ai4as_pitfalls_in_machine_learning_2026/requirements.txt


In [3]:
# Optional: install exactly the pinned minimal requirements generated in Cell 1.
# %pip install -r ../requirements.txt